# Fabric Local Development - Getting Started

This notebook demonstrates the complete data pipeline workflow:
1. Connect to SQL Server (data source)
2. Read data with PySpark
3. Transform data (like you would in Fabric)
4. Write results back to SQL Server (data target)

**Environment:** Local Docker containers mimicking Microsoft Fabric architecture

## Step 1: Initialize PySpark Session

Create a Spark session connected to our Spark cluster

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, round
import os

# Create Spark session with SQL Server JDBC driver
spark = SparkSession.builder \
    .appName("FabricLocalDev") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print(f"✅ Spark Session Created!")
print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")

## Step 2: Configure SQL Server Connection

Set up connection parameters (stored in environment variables)

In [ ]:
# SQL Server connection details (from docker-compose.yml environment variables)
sql_server_host = os.getenv('SQL_SERVER_HOST', 'sqlserver')
sql_server_port = os.getenv('SQL_SERVER_PORT', '1433')
sql_server_user = os.getenv('SQL_SERVER_USER', 'sa')
sql_server_password = os.getenv('SQL_SERVER_PASSWORD', 'YourStrong@Passw0rd')
database_name = 'FabricDev'

# JDBC connection string
jdbc_url = f"jdbc:sqlserver://{sql_server_host}:{sql_server_port};databaseName={database_name};encrypt=false;trustServerCertificate=true"

# Connection properties
connection_properties = {
    "user": sql_server_user,
    "password": sql_server_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print(f"✅ SQL Server connection configured")
print(f"Host: {sql_server_host}:{sql_server_port}")
print(f"Database: {database_name}")

## Step 3: Read Data from SQL Server

Load the Sales table into a Spark DataFrame

In [ ]:
# Read Sales table from SQL Server
df_sales = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.Sales",
    properties=connection_properties
)

print(f"✅ Data loaded from SQL Server")
print(f"Total rows: {df_sales.count()}")
print(f"\nSchema:")
df_sales.printSchema()
print(f"\nSample data (first 5 rows):")
df_sales.show(5, truncate=False)

## Step 4: Data Exploration

Explore the data with PySpark transformations

In [ ]:
# Basic statistics
print("📊 Sales Summary Statistics:")
df_sales.select("Quantity", "UnitPrice", "TotalAmount").describe().show()

# Count by region
print("\n🌍 Sales by Region:")
df_sales.groupBy("Region").count().orderBy(col("count").desc()).show()

# Top products by revenue
print("\n🏆 Top Products by Revenue:")
df_sales.groupBy("ProductName") \
    .agg(sum("TotalAmount").alias("TotalRevenue")) \
    .orderBy(col("TotalRevenue").desc()) \
    .show()

## Step 5: Data Transformation

Create aggregated sales summary by region (like a Fabric data pipeline)

In [ ]:
# Aggregate sales by region
df_summary = df_sales.groupBy("Region").agg(
    round(sum("TotalAmount"), 2).alias("TotalSales"),
    count("*").alias("TotalOrders"),
    round(avg("TotalAmount"), 2).alias("AvgOrderValue")
)

print("✅ Sales summary created")
df_summary.show()

# Show the transformation plan (like Fabric's execution plan)
print("\n📋 Execution Plan:")
df_summary.explain()

## Step 6: Write Results Back to SQL Server

Save transformed data to the SalesSummary table

In [ ]:
# Write to SQL Server (overwrite mode)
df_summary.write.jdbc(
    url=jdbc_url,
    table="dbo.SalesSummary",
    mode="overwrite",  # Options: overwrite, append, ignore, error
    properties=connection_properties
)

print("✅ Data written to SQL Server!")
print("Table: dbo.SalesSummary")

# Verify by reading back
df_verify = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.SalesSummary",
    properties=connection_properties
)

print("\n📊 Verified data in SQL Server:")
df_verify.show()

## Step 7: Working with Pandas (Optional)

Convert Spark DataFrame to Pandas for visualization or analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert to Pandas
pdf_summary = df_summary.toPandas()

print("✅ Converted to Pandas DataFrame")
print(pdf_summary)

# Simple visualization
plt.figure(figsize=(10, 6))
plt.bar(pdf_summary['Region'], pdf_summary['TotalSales'])
plt.title('Total Sales by Region')
plt.xlabel('Region')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n📈 Chart created!")

## Step 8: SQL Query Execution (Alternative Approach)

Use SQL directly instead of DataFrame API

In [ ]:
# Register DataFrame as temporary view
df_sales.createOrReplaceTempView("sales_view")

# Run SQL query
sql_query = """
SELECT 
    SalesRep,
    COUNT(*) as TotalOrders,
    ROUND(SUM(TotalAmount), 2) as TotalRevenue,
    ROUND(AVG(TotalAmount), 2) as AvgOrderValue
FROM sales_view
GROUP BY SalesRep
ORDER BY TotalRevenue DESC
"""

df_sales_rep = spark.sql(sql_query)

print("✅ SQL query executed")
print("\n👥 Sales Performance by Rep:")
df_sales_rep.show()

## Step 9: Working with Files

Read/write CSV and Parquet files (common in Fabric)

In [ ]:
# Export to CSV
df_summary.coalesce(1).write.mode("overwrite").option("header", "true").csv("/opt/data/sales_summary.csv")
print("✅ Exported to CSV: /opt/data/sales_summary.csv")

# Export to Parquet (columnar format, used by Fabric internally)
df_sales.write.mode("overwrite").parquet("/opt/data/sales_data.parquet")
print("✅ Exported to Parquet: /opt/data/sales_data.parquet")

# Read back from Parquet
df_from_parquet = spark.read.parquet("/opt/data/sales_data.parquet")
print(f"\n✅ Read from Parquet: {df_from_parquet.count()} rows")
df_from_parquet.show(5)

## Step 10: Clean Up

Stop the Spark session when done

In [ ]:
# spark.stop()
# print("✅ Spark session stopped")

print("ℹ️ Spark session still active. Uncomment above lines to stop.")

---

## 🎉 Congratulations!

You've completed a full data pipeline workflow:
- ✅ Connected to SQL Server
- ✅ Read data with PySpark
- ✅ Transformed data
- ✅ Wrote results back to SQL Server
- ✅ Worked with files (CSV, Parquet)

### Next Steps:
1. Modify this notebook for your own data
2. Create more complex transformations
3. Add your own data sources
4. Experiment with Spark SQL

### Useful Resources:
- Spark UI: http://localhost:8080
- Jupyter Lab: http://localhost:8888
- SQL Server: localhost:1433

### Tips:
- Use `df.printSchema()` to see DataFrame structure
- Use `df.explain()` to see execution plan
- Use `spark.sql()` for SQL queries
- Check Spark UI for job monitoring